# Wallet Signal v2

Phase 2+3 (skill path): compute wallet skill metrics, select wallets, build signal events, calibrate and score.

In [1]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
# END_DATE_TRAIN and TRAIN_A_END_DATE are derived from the data below
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

COHORT_TOP_N_DEFAULT = 100

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
SELECTED_WALLETS_PATH   = WORKSPACE_DIR / "selected_wallets_v2.parquet"
SELECTED_PROFILES_PATH  = WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


In [3]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
del _sample
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-03-07  TRAIN_A_END_DATE=2026-02-16


In [4]:
# ── Phase 2: compute / load wallet skill metrics ──────────────────────────────
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,45407
train_b_wallets,45407
full_train_wallets,45407
test_wallets,45407


In [5]:
# ── Phase 2: cohort selection sweep ─────────────────────────────────────────
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
0,prob_edge_shrunk,50,50,2188,0.084714,0.144695,0.901704,100566.740631,0.417733
1,prob_edge_shrunk,100,100,3418,-0.005363,0.103747,0.597762,32786.499393,0.447630
2,prob_edge_shrunk,200,200,10178,0.000995,0.084642,0.438224,72766.808891,0.470132
13,avg_copy_roi_capped,300,300,9398,0.030304,0.066977,0.410752,174619.921275,0.433922
12,avg_copy_roi_capped,200,200,4473,0.036930,0.059738,0.384170,128404.802915,0.418735
3,prob_edge_shrunk,300,300,13101,0.012600,0.079378,0.370006,166870.171841,0.496756
14,avg_copy_roi_capped,500,500,13994,0.027761,0.042974,0.324798,260347.157546,0.454838
10,avg_copy_roi_capped,50,50,2141,0.015695,0.051842,0.307529,19774.525120,0.410556
11,avg_copy_roi_capped,100,100,2705,0.025123,0.047399,0.266803,51251.827690,0.419593
4,prob_edge_shrunk,500,500,26621,0.029551,0.061264,0.222028,640161.293125,0.513805


In [6]:
# ── pick best metric / top-N ─────────────────────────────────────────────────
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'prob_edge_shrunk', 'best_top_n': 50}

In [7]:
# ── Phase 2: select wallets ───────────────────────────────────────────────────
selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(SELECTED_WALLETS_PATH, index=False)
selected_wallets[["wallet", "open_buy_trades", "distinct_markets",
                   "recent_open_buy_trades", BEST_SELECTION_METRIC, "wallet_quality"]].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,prob_edge_shrunk,wallet_quality
0,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,532,506,529,0.304533,1.00
1,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,772,743,769,0.300616,0.98
2,0x27b713fc1c89d498f19977c8bc7788a161fb7710,787,23,787,0.294186,0.96
3,0x4d7fad0c5944fc24d4a67110f8e31abd5f559485,218,210,170,0.290809,0.94
4,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,208,199,208,0.285536,0.92
5,0x82307f44c9405e73dc1cff466073dcc505535121,389,368,389,0.261187,0.90
6,0xb9bca9fa4069228a37b583d64ff75efdf36b3498,21,21,21,0.248022,0.88
7,0xf1f06f49be8ce5681752ae80e660aeaace6858df,308,302,92,0.235090,0.86
8,0xc178402031235263f78c1a43bba8cd49d2be35b3,119,114,12,0.220636,0.84
9,0xfd4263b3ad08226034fe1b1ea678a46d80b58895,209,203,209,0.219044,0.82


In [8]:
# ── Phase 3: build wallet profiles and signal events ─────────────────────────
from polymarket_analysis.signal.builder import (
    build_wallet_profiles,
    build_signal_events,
)

if SELECTED_PROFILES_PATH.exists():
    selected_wallet_profiles = pd.read_parquet(SELECTED_PROFILES_PATH)
else:
    selected_wallet_profiles = build_wallet_profiles(
        dataset, selected_wallets, period="full_train",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
    )
    selected_wallet_profiles.to_parquet(SELECTED_PROFILES_PATH, index=False)

if CALIBRATION_SIGNALS_PATH.exists():
    _, train_b_open_buys = None, pd.read_parquet(CALIBRATION_SIGNALS_PATH)
else:
    _, train_b_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="train_b",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    train_b_open_buys.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

if TEST_SIGNALS_PATH.exists():
    test_open_buys = pd.read_parquet(TEST_SIGNALS_PATH)
else:
    _, test_open_buys = build_signal_events(
        dataset, selected_wallet_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    test_open_buys.to_parquet(TEST_SIGNALS_PATH, index=False)

print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 2,188  test signals: 2,936


In [9]:
# ── Phase 3: calibrate score tables on train-B ───────────────────────────────
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# score distribution for display
score_deciles = (
    train_b_scored.assign(score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop"))
    .groupby("score_decile")
    .agg(
        count=("signal_score", "size"),
        avg_signal_score=("signal_score", "mean"),
        avg_copy_roi_capped=("copy_roi_capped", "mean"),
    )
    .reset_index()
)
score_deciles

Global edge: 0.1447
Selected signal threshold: 0.80


,score_decile,count,avg_signal_score,avg_copy_roi_capped
0,0,219,0.444743,1.860643
1,1,219,0.501713,1.754792
2,2,219,0.576948,1.051924
3,3,218,0.626133,0.362047
4,4,219,0.659588,0.493976
5,5,219,0.697506,0.589540
6,6,226,0.740450,0.394098
7,7,214,0.784169,0.830189
8,8,216,0.840562,1.673228
9,9,219,0.931566,0.029301


In [10]:
# ── Phase 3: apply score to test signals ─────────────────────────────────────
from polymarket_analysis.signal.scorer import apply_signal_score

test_scored = apply_signal_score(test_open_buys, price_table, consensus_table)
print(f"Scored test signals: {len(test_scored):,}")
test_scored[["wallet", "signal_score", "price", "condition_id"]].head(10)


Scored test signals: 2,936


,wallet,signal_score,price,condition_id
0,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,0.831315,0.10,0x415a8023c79aa96d4d9eff7854c93f278fb16f8de244...
1,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,0.940329,0.10,0xf8d309dac63267975cb7e0312759a5a06d20e248b848...
2,0x3c593aeb73ebdadbc9ce76d4264a6a2af4011766,0.552481,0.35,0xa6c8504a9b933f1cc856cc9a49517d976aae56c39d20...
3,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,0.796534,0.08,0x5e42739c2cbe955bca3cedeb2f20f213cf155819148f...
4,0x82307f44c9405e73dc1cff466073dcc505535121,0.840481,0.28,0xcf2e532022a77d44b23c94338380a4a139ea25e63a4f...
5,0x4247c89f4a9236a6b5fb24502023bcc98ab1455f,0.656691,0.41,0x633b744818a4fd6c58c9daeff8f27e77a2f28d7f1165...
6,0x48ecc049f171a8c7758e5d351e5f6c3c6df58f36,0.616917,0.41,0x633b744818a4fd6c58c9daeff8f27e77a2f28d7f1165...
7,0xfd4263b3ad08226034fe1b1ea678a46d80b58895,0.660740,0.12,0x633b744818a4fd6c58c9daeff8f27e77a2f28d7f1165...
8,0x82307f44c9405e73dc1cff466073dcc505535121,0.840481,0.28,0xcf2e532022a77d44b23c94338380a4a139ea25e63a4f...
9,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,0.763464,0.13,0x4202cb98355daa8d0b8585f700b73ad2af385a354feb...


In [11]:
# ── Phase 3: build wallet cohorts ────────────────────────────────────────────
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets,
    top_n=BEST_TOP_N,
)
{name: len(df) for name, df in wallet_cohorts.items()}


{'quality_core': 50, 'early_entry': 32, 'smooth_pnl': 50}

In [12]:
# ── Persist wallet cohorts ────────────────────────────────────────────────────
from polymarket_analysis.wallet_selection.persistence import WalletSet, save_wallet_set

for cohort_name, cohort_df in wallet_cohorts.items():
    ws = WalletSet(id=cohort_name, wallets=cohort_df)
    parquet_path, json_path = save_wallet_set(ws, WORKSPACE_DIR)
    print(f"Saved cohort '{cohort_name}': {len(cohort_df)} wallets  -> {parquet_path.name}")

Saved cohort 'quality_core': 50 wallets  -> quality_core.parquet
Saved cohort 'early_entry': 32 wallets  -> early_entry.parquet
Saved cohort 'smooth_pnl': 50 wallets  -> smooth_pnl.parquet
